# Stage 1 & 2A: Data Cleaning, Missingness Analysis & Univariate EDA
### AMEO 2015 Engineering Placement Benchmark (2,998 Records, 34 Features)
**Assigned to:** Dhruven  
**Course:** IT462 Exploratory Data Analysis | Syllabus Modules 1, 2, 4, 5, 6, 7  

---
## 📌 Syllabus Modules Covered in this Notebook
1. **Module 1: Data and Statistics** (Data Science Pipeline, Role of EDA in ML, Structured vs Unstructured, Population vs Sample)
2. **Module 2: Measurement Scales** (Nominal, Ordinal, Interval, Ratio, Discrete vs Continuous Variables)
3. **Module 4: Data Import & Cleaning** (Inspection, Insights, Duplicate Detection, Inconsistent Formats, Invalid Entries, Type Conversion)
4. **Module 5: Missing Data Analysis** (MCAR, MAR, MNAR Mechanisms, Missingness Visualization, Missing Data Treatment)
5. **Module 6: Descriptive Summary** (Central Tendency, Dispersion, MAD, Coefficient of Variation, Skewness, Kurtosis, Tukey's Five-Number Summary)
6. **Module 7: Univariate Analysis** (Histograms, Density Plots / KDE, Boxplots, Violin Plots, Bar Charts)


## 1. Setup & Environment Configuration
Import essential numerical and statistical libraries (`numpy`, `pandas`, `scipy.stats`, `matplotlib`, `seaborn`, `missingno`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Optional missingness visualization library
try:
    import missingno as msno
except ImportError:
    msno = None

# Styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Environment configured successfully.")
if msno:
    print("missingno library loaded.")
else:
    print("missingno not installed; fallback to seaborn missingness heatmaps.")

## 2. Foundations: Data and Statistics (Syllabus Module 1)
### Statistical Definitions & Taxonomy
- **What is Data?** Facts, figures, or information collected together for reference or analysis. In our context, data comprises academic records, cognitive psychometrics, and employment outcomes for engineering graduates.
- **The Data Science Pipeline:**
  1. *Collection:* Sourcing real empirical records (Aspiring Minds AMCAT study).
  2. *Data Cleaning:* Rectifying anomalies, sentinel codes (`-1`), missing values, and formatting discrepancies.
  3. *Exploratory Data Analysis (EDA):* Summarizing distributional shapes, detecting outliers, computing correlation, and discovering patterns.
  4. *Feature Engineering & Transformation:* Correcting skewness, encoding categorical tiers, standardizing scales.
  5. *Predictive Modeling:* Supervised learning benchmarks.
  6. *Data Storytelling & Dashboards:* Communicating actionable insights to university leadership.
- **Role of EDA in Machine Learning:** Proposed by John Tukey (1977), EDA serves as the foundation before formal modeling to understand data characteristics, check statistical assumptions (normality, multicollinearity, homoscedasticity), formulate hypotheses, and prevent "garbage in, garbage out".
- **Structured vs Unstructured Data:** Our AMEO dataset is purely **Structured Tabular Data** ($N = 2,998$ records $\times$ 34 well-defined attribute columns with rigid relational schema), contrasted with unstructured data (e.g., student resumes as PDFs, interview audio, coding video recordings).
- **Population vs Sample:**
  - *Target Population:* All ~1.5 million engineering graduates graduating annually across all Indian AICTE institutions.
  - *Empirical Sample:* $n = 2,998$ engineering graduates who took the standardized AMCAT assessment.

## 3. Data Import & Initial Inspection (Syllabus Module 4)
> **TODO (Dhruven):** Load `../data/raw/Engineering_graduate_salary.csv`, inspect `.shape`, `.info()`, `.head()`, and examine data types.

In [ ]:
raw_path = '../data/raw/Engineering_graduate_salary.csv'
df = pd.read_csv(raw_path)
print(f"Loaded Raw Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.info()
df.head()

## 4. Measurement Scales & Variable Types (Syllabus Module 2)
Categorizing the 34 attributes according to Stevens' (1946) Four Measurement Scales and Discrete vs Continuous nature:

| Scale | Mathematical Properties | AMEO Dataset Features | Permissible Statistics |
| :--- | :--- | :--- | :--- |
| **Nominal** | Identity, equality ($=, \neq$) | `Gender`, `10board`, `12board`, `Degree`, `Specialization`, `CollegeState`, `JobCity` | Mode, Frequency, Contingency Tables, Chi-Square |
| **Ordinal** | Order, rank ($>, <$) | `CollegeTier` (1 vs 2), `CollegeCityTier` (0 vs 1) | Median, Percentiles, Rank Correlation (Spearman) |
| **Interval** | Meaningful differences ($+, -$), arbitrary zero | `GraduationYear`, `12graduation`, `conscientiousness`, `agreeableness`, `extraversion`, `nueroticism`, `openess_to_experience` (Standardized z-scores) | Mean, Standard Deviation, Pearson Correlation |
| **Ratio** | True absolute zero, meaningful ratios ($\times, \div$) | `Salary` (INR), `collegeGPA`, `10percentage`, `12percentage`, `English`, `Logical`, `Quant`, `Domain`, `ComputerProgramming` | Geometric Mean, Coefficient of Variation (CV) |

### Discrete vs Continuous Variables:
- **Discrete Variables:** Finite or countably infinite values (e.g., `GraduationYear`, `CollegeTier`, `CollegeCityTier`).
- **Continuous Variables:** Uncountably infinite values within an interval (e.g., `Salary`, `collegeGPA`, `10percentage`, `Quant`, `agreeableness`).

In [ ]:
# TODO (Dhruven): Verify and print variables grouped by measurement scale
scales = {
    'Nominal': ['Gender', '10board', '12board', 'Degree', 'Specialization', 'CollegeState'],
    'Ordinal': ['CollegeTier', 'CollegeCityTier'],
    'Interval': ['conscientiousness', 'agreeableness', 'extraversion', 'nueroticism', 'openess_to_experience', 'GraduationYear'],
    'Ratio': ['Salary', 'collegeGPA', '10percentage', '12percentage', 'English', 'Logical', 'Quant', 'Domain', 'ComputerProgramming']
}

for scale, cols in scales.items():
    present = [c for c in cols if c in df.columns]
    print(f"{scale} Scale ({len(present)} features): {present}")

## 5. Data Cleaning, Validation & Datetime Parsing (Syllabus Module 4 & Lecture Notes)
### Pipeline Sequence from Lecture Notes (Slide 1):
1. **Step 1: Import Data** (`pd.read_csv`)
2. **Step 2: Validation / Prescreening** (Shape, null detection, invalid codes)
3. **Step 2b: Time Series / Datetime Parsing & Timedelta:**
   - Parse date string features: `df['DOB_parsed'] = pd.to_datetime(df['DOB'])`
   - Compute timedelta features: $\text{Age at Graduation} = \text{GraduationYear} - \text{DOB Year}$
4. **Step 3: Cleaning & Pre-processing:**
   - Duplicate detection (`df.duplicated().sum()`)
   - Invalid & Sentinel codes: Handle `-1` unattempted optional technical module scores (`CivilEngg`, `MechanicalEngg`, `ComputerProgramming`).

> **TODO (Dhruven):** Implement duplicate checks, datetime parsing on `DOB`, and sentinel `-1` handling.

In [ ]:
# TODO (Dhruven): Step 2 & 2b from Lecture Notes
duplicates = df.duplicated().sum()
print(f"Step 2 Validation - Duplicate Rows Found: {duplicates}")

# Step 2b: Datetime parsing & Timedelta calculation
if 'DOB' in df.columns:
    df['DOB_parsed'] = pd.to_datetime(df['DOB'], errors='coerce')
    df['Age_at_Graduation'] = df['GraduationYear'] - df['DOB_parsed'].dt.year
    print(f"Step 2b - Parsed DOB to Datetime. Sample Ages: {df['Age_at_Graduation'].head().tolist()}")

# Step 3: Inspect sentinel value (-1) in technical domain scores
tech_cols = ['Domain', 'ComputerProgramming', 'ElectronicsAndSemicon', 'ComputerScience', 'MechanicalEngg', 'ElectricalEngg', 'CivilEngg']
for col in tech_cols:
    if col in df.columns:
        count_neg1 = (df[col] == -1).sum()
        print(f"{col}: {count_neg1} students with -1 (unattempted module: {count_neg1/len(df)*100:.1f}%)")

## 6. Missing Data Analysis: MCAR, MAR, MNAR (Syllabus Module 5)
### Missingness Mechanisms:
1. **MCAR (Missing Completely at Random):** Missingness is entirely independent of observed and unobserved data (e.g., random network packet drop during test upload).
2. **MAR (Missing at Random):** Missingness depends systematically on observed attributes but not on the missing value itself (e.g., Mechanical Engineering students did not take the `ComputerScience` module because of their `Specialization`).
3. **MNAR (Missing Not at Random):** Missingness depends directly on the unobserved value itself (e.g., students who performed poorly in an optional test chose not to submit their score).

> **TODO (Dhruven):** Analyze null counts, visualize missingness matrix or heatmap, and state the missingness mechanism.

In [ ]:
# TODO (Dhruven): Check explicit nulls and missingness
print("Explicit Null Counts:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Missingness visualization
if msno:
    plt.figure(figsize=(10, 4))
    msno.matrix(df[tech_cols].replace(-1, np.nan), sparkline=False)
    plt.title("Missingness Matrix for Technical Domain Modules (MAR Pattern)")
    plt.show()
else:
    plt.figure(figsize=(10, 4))
    sns.heatmap(df[tech_cols].replace(-1, np.nan).isnull(), cbar=False, cmap='viridis')
    plt.title("Missingness Heatmap for Technical Modules")
    plt.show()

## 7. Descriptive Summaries: Shape, Dispersion & Central Tendency (Syllabus Module 6)
Compute rigorous statistical descriptors across all continuous variables:
- **Measures of Central Tendency:** Mean, Median, Mode
- **Measures of Dispersion / Variation:** Variance ($\sigma^2$), Standard Deviation ($\sigma$), Interquartile Range ($IQR = Q_3 - Q_1$)
- **Median Absolute Deviation (MAD):** $\text{MAD} = \text{median}(|X_i - \tilde{X}|)$ — Tukey's robust outlier-resistant dispersion metric
- **Coefficient of Variation (CV):** $CV = \frac{\sigma}{\mu} \times 100\%$ — scale-independent relative dispersion
- **Measures of Shape:** Fisher-Pearson Skewness (asymmetry) & Kurtosis (tailedness/heaviness of tails)
- **Tukey's Five-Number Summary:** Minimum, First Quartile ($Q_1$), Median, Third Quartile ($Q_3$), Maximum

> **TODO (Dhruven):** Build a consolidated descriptive statistics table.

In [ ]:
# TODO (Dhruven): Compute consolidated statistical summary table
numeric_cols = ['Salary', 'collegeGPA', '10percentage', '12percentage', 'English', 'Logical', 'Quant']

stats_summary = []
for col in numeric_cols:
    if col in df.columns:
        data = df[col].dropna()
        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        iqr = q3 - q1
        mad = np.median(np.abs(data - data.median()))
        cv = (data.std() / data.mean()) * 100 if data.mean() != 0 else np.nan
        skew = stats.skew(data)
        kurt = stats.kurtosis(data)
        
        stats_summary.append({
            'Variable': col,
            'Mean': round(data.mean(), 2),
            'Median': round(data.median(), 2),
            'Std Dev': round(data.std(), 2),
            'Variance': round(data.var(), 2),
            'IQR': round(iqr, 2),
            'MAD': round(mad, 2),
            'CV (%)': round(cv, 2),
            'Skewness': round(skew, 2),
            'Kurtosis': round(kurt, 2),
            'Min': round(data.min(), 2),
            'Q1': round(q1, 2),
            'Q3': round(q3, 2),
            'Max': round(data.max(), 2)
        })

summary_df = pd.DataFrame(stats_summary)
summary_df

## 8. Univariate Visualizations (Syllabus Module 7)
Visualizing the individual distributions using the 5 core univariate techniques specified in the syllabus:
1. **Histogram:** Frequency distribution of `Salary` (identifying right skewness).
2. **Density Plot (KDE):** Continuous empirical density estimation of `collegeGPA` and `Quant`.
3. **Boxplot:** Tukey boxplot displaying median, whiskers, and extreme outliers.
4. **Violin Plot:** Combining KDE density shape with boxplot quartiles.
5. **Bar Chart:** Frequency breakdown of categorical attributes (`Gender`, `CollegeTier`, `Degree`).

> **TODO (Dhruven):** Plot univariate figures and save key plots to `../reports/figures/`.

In [ ]:
# TODO (Dhruven): Generate univariate plots
os.makedirs('../reports/figures', exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogram: Salary
sns.histplot(df['Salary'], bins=30, kde=True, ax=axes[0, 0], color='royalblue')
axes[0, 0].set_title('Salary Distribution (Histogram + KDE)')
axes[0, 0].set_xlabel('Annual Salary (INR)')

# 2. Boxplot: collegeGPA
sns.boxplot(x=df['collegeGPA'], ax=axes[0, 1], color='mediumseagreen')
axes[0, 1].set_title('College GPA Distribution (Tukey Boxplot)')

# 3. Violin Plot: Quant Score
sns.violinplot(x=df['Quant'], ax=axes[1, 0], color='orchid')
axes[1, 0].set_title('Quant Score Distribution (Violin Plot)')

# 4. Bar Chart: College Tier Breakdown
tier_counts = df['CollegeTier'].value_counts()
axes[1, 1].bar(['Tier 2', 'Tier 1'], [tier_counts.get(2, 0), tier_counts.get(1, 0)], color=['coral', 'cornflowerblue'])
axes[1, 1].set_title('College Tier Breakdown (Bar Chart)')
axes[1, 1].set_ylabel('Student Count')

plt.tight_layout()
plt.savefig('../reports/figures/dhruven_univariate_eda.png', dpi=300)
plt.show()

## 9. Export Cleaned Dataset for Stage 2B (Prit)
> **TODO (Dhruven):** Once cleaning and invalid code handling is finalized, export the clean dataset to `../data/processed/ameo_cleaned.csv`.

In [ ]:
# TODO (Dhruven): Uncomment to export after completing Stage 1 & 2A cleaning
# os.makedirs('../data/processed', exist_ok=True)
# df.to_csv('../data/processed/ameo_cleaned.csv', index=False)
# print("Cleaned dataset saved successfully to ../data/processed/ameo_cleaned.csv")